In [1]:
import xarray as xr
import pandas as pd
import numpy as np
from pathlib import Path
import datetime
from src.get_rda_era5 import ERA5DataSource
import re
import zarr

In [2]:
lat_dict = {
    'full': slice(50, 25),
    'small': slice(45, 30),
    'slgt_small': slice(50, 25),
    'slgt_full': slice(50, 25)
}

lon_dict = {
    'full': slice(360-125, 360-66),
    'small': slice(360-105, 360-85),
    'slgt_small': slice(360-125, 360-66),
    'slgt_full': slice(360-125, 360-66)
}

levels_dict = {
    'full': [925, 850, 700, 500, 300],
    'small': [925, 850, 700, 500, 300],
    'slgt_small': [925, 850, 700, 500, 300],
    'slgt_full': [925, 850, 700, 500, 300]
}

time_thin_dict = {
    'full': 1,
    'small': 6,
    'slgt_small': 6,
    'slgt_full': 2,
}

space_thin_dict = {
    'full': 1,
    'small': 4,
    'slgt_small': 4,
    'slgt_full': 1
}

risk_level_dict = {
    'full': ['MDT', 'HIGH'],
    'small': ['MDT', 'HIGH'],
    'slgt_small': ['SLGT', 'ENH', 'MDT', 'HIGH'],
    'slgt_full': ['SLGT', 'ENH', 'MDT', 'HIGH']
}

pressure_var_dict = {
    'full': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'small': ["geopotential", "specific_humidity", "temperature", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'slgt_small': ["geopotential", "specific_humidity", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"],
    'slgt_full': ["geopotential", "specific_humidity", "u_component_of_wind", "v_component_of_wind", "vertical_velocity"]
}

surface_var_dict = {
    'full': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'small': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'slgt_small': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"],
    'slgt_full': ["10m_u_component_of_wind", "10m_v_component_of_wind", "2m_dewpoint_temperature", "2m_temperature"]
}

In [4]:
detail = 'full'

In [5]:
# --- risk days
pph = xr.load_dataset("data/raw_data/labelled_pph.nc")
missing_dates = [
    '200204250000', '200208300000', '200304150000', '200304160000',
    '200306250000', '200307270000', '200307280000', '200312280000',
    '200404140000', '200408090000', '200905280000', '201105210000',
    '202005240000', '200510240000'
]
dates_of_interest = pph["time"][pph["MAX_CAT"].isin(risk_level_dict[detail])]
dates_of_interest = dates_of_interest[dates_of_interest > "200203310000"]
dates_of_interest = dates_of_interest[~(dates_of_interest.isin(missing_dates))]
selected_days = pd.to_datetime(dates_of_interest.values, format="%Y%m%d%H%M").normalize()

years = np.unique(selected_days.year)

In [6]:
LONG_TO_SHORT = {
    "10m_u_component_of_wind": "10u",
    "10m_v_component_of_wind": "10v",
    "2m_temperature": "2t",
    "2m_dewpoint_temperature": "2d",
    # "geopotential_at_surface": "z",
    # "toa_incident_solar_radiation": "tisr",
    # PRESSURE LEVEL VARS
    "geopotential": "z",
    # "potential_vorticity": "pv",
    "specific_humidity": "q",
    "temperature": "t",
    "u_component_of_wind": "u",
    "v_component_of_wind": "v",
    "vertical_velocity": "w"
}

SHORT_TO_LONG = {value: key for key, value in LONG_TO_SHORT.items()}


def longname_to_channel(longname, level=None):
    """
    Return a channel string e.g. "u10m", "t850", "z500".
    If longname corresponds to a surface variable (10m, 2m, etc) no level needed.
    If it is a pressure-variable and level given -> return e.g. "t850".
    """
    # surface pattern
    if longname in LONG_TO_SHORT:
        short = LONG_TO_SHORT[longname]
        if level is None:
            return short
        else:
            return f"{short}{int(level)}"
    raise KeyError(f"No conversion known for {longname}; add to LONG_TO_SHORT")


def channels_for_config(cfg):
    return (
        # surface variables (single level)
        [longname_to_channel(v)
         for v in surface_var_dict[cfg]
         if LONG_TO_SHORT.get(v, "")]

        +

        # pressure-level variables (expanded over levels)
        [longname_to_channel(v, lvl)
         for v in pressure_var_dict[cfg]
         if LONG_TO_SHORT.get(v, "")
         for lvl in levels_dict[cfg]]
    )

In [7]:
# prep channel names
all_channels = channels_for_config(detail)

In [8]:
def collect_one_day(ds, day, detail):
    """
    Returns a DataArray:
      (day=1, tod, channel, lat, lon)
    """
    tods = list(range(0, 24, time_thin_dict[detail]))
    per_tod = []

    for tod in tods:
        # print(tod)
        da = ds[day.to_pydatetime() + datetime.timedelta(hours=tod + 12)]  # (time=1, channel, lat, lon)

        da = (
            da
            .isel(time=0, drop=True)
            .sel(latitude=lat_dict[detail], longitude=lon_dict[detail])
            .thin({"latitude": space_thin_dict[detail], "longitude": space_thin_dict[detail]})
            .expand_dims(tod=[tod])   # set tod as a 1-length dimension with coordinate
        )
        per_tod.append(da)

    block = xr.concat(per_tod, dim="tod")  # (tod, channel, lat, lon)

    # day is just the normalized date you started with (12Z–12Z convention is implicit in the sampling)
    day0 = pd.Timestamp(day).normalize().to_datetime64()
    block = block.expand_dims(day=[day0])  # (day=1, tod, channel, lat, lon)

    return block.transpose("day", "tod", "channel", "latitude", "longitude")



def channel_da_to_dataset(da):
    """
    Convert channel-stacked DataArray into final Dataset.

    """

    ds_out = {}

    for ch in da.channel.values:
        sub = da.sel(channel=ch).drop_vars("channel")

        if bool(re.search(r'\d{3}$', ch)):
            level = ch[-3:]
            longname = SHORT_TO_LONG[ch[:-3]]
        else:
            level = None
            longname = SHORT_TO_LONG[ch]

        if level is None:
            # surface variable
            ds_out.setdefault(longname, []).append(sub)
        else:
            # pressure-level variable
            sub = sub.assign_coords(level=level).expand_dims("level")
            ds_out.setdefault(longname, []).append(sub)

    data_vars = {}

    for name, pieces in ds_out.items():
        merged = xr.concat(pieces, dim="level") if "level" in pieces[0].dims else pieces[0]
        data_vars[name] = merged

    return xr.Dataset(data_vars)

In [ ]:
out_dir = Path(f"/glade/work/milesep/convective_outlook_ml/inputs_raw_{detail}_glade.zarr")

# Skip everything except the most recent day in the store (always redo the last day). Can pick up where it left off
done = set()
if out_dir.exists():
    d = np.asarray(xr.open_zarr(out_dir, consolidated=False)["day"].values).astype("datetime64[ns]")
    if d.size:
        done = set(d[d < d.max()])

ds = ERA5DataSource(all_channels)
store_exists = out_dir.exists()

for day in selected_days:
    d64 = np.datetime64(day.to_datetime64()).astype("datetime64[ns]")
    if d64 in done:
        continue

    print(f"Processing {day.date()}")
    day_ds = channel_da_to_dataset(collect_one_day(ds, day, detail))

    day_ds.to_zarr(
        out_dir,
        mode="a" if store_exists else "w",
        append_dim="day" if store_exists else None,
        consolidated=False,
    )
    store_exists = True  # after first successful write

zarr.consolidate_metadata(out_dir)

Processing 2002-04-16
Processing 2002-04-18
Processing 2002-04-20
Processing 2002-04-21
Processing 2002-04-23
Processing 2002-04-27
Processing 2002-04-28
Processing 2002-05-01
Processing 2002-05-05
